Instalimi dhe importimi i librarive të nevojshme

In [ ]:
pip install pandas

In [ ]:
import pandas as pd
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

Mbledhja e të dhënave, definimi i tipeve të dhënave, kualiteti i të dhënave

In [14]:
df = pd.read_csv(r'./master.csv')

print(df.dtypes)
print(df.describe())

country                object
year                    int64
sex                    object
age                    object
suicides_no             int64
population              int64
suicides/100k pop     float64
country-year           object
HDI for year          float64
 gdp_for_year ($)      object
gdp_per_capita ($)      int64
generation             object
dtype: object
               year   suicides_no    population  suicides/100k pop  \
count  27820.000000  27820.000000  2.782000e+04       27820.000000   
mean    2001.258375    242.574407  1.844794e+06          12.816097   
std        8.469055    902.047917  3.911779e+06          18.961511   
min     1985.000000      0.000000  2.780000e+02           0.000000   
25%     1995.000000      3.000000  9.749850e+04           0.920000   
50%     2002.000000     25.000000  4.301500e+05           5.990000   
75%     2008.000000    131.000000  1.486143e+06          16.620000   
max     2016.000000  22338.000000  4.380521e+07         224.970000

Riemerimi i kolonave për përdorim më të lehtë

In [15]:
df=df.rename(columns={'sex':'gender','gdp_per_capita ($)':'gdp_per_capita',' gdp_for_year ($) ':'gdp_for_year', 'HDI for year' : 'hdi_for_year'})

Modifikimi i tipit te te dhenave per kolonen 'gdp_for_year'

In [16]:
df['gdp_for_year'] = df['gdp_for_year'].str.replace(',', '').astype(int)

Ndryshimi i dimensionalitetit

In [17]:
df = df[['country','year', 'gender', 'age', 'suicides_no','population','suicides/100k pop','gdp_for_year','gdp_per_capita','hdi_for_year']]

Menaxhimi i vlerave null

In [18]:
df = df.sort_values(by=['country', 'year'])

# Forward fill and backfill within each country
df['hdi_for_year'] = df.groupby('country')['hdi_for_year'].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))

# Calculate yearly mean for each country
yearly_mean = df.groupby(['country', 'year'])['hdi_for_year'].mean().reset_index()

# Interpolate mean values for each country
yearly_mean['hdi_for_year'] = yearly_mean.groupby('country')['hdi_for_year'].transform(lambda x: x.interpolate(method='linear'))

# Merge back the means into the original DataFrame
df = df.merge(yearly_mean, on=['country', 'year'], suffixes=('', '_mean'))

# Combine original HDI with the interpolated mean
df['hdi_for_year'] = df['hdi_for_year'].combine_first(df['hdi_for_year_mean'])

df = df.drop(columns=['hdi_for_year_mean'])

Mostrimi i të dhënave (10%)

In [20]:
sampled_data = df.sample(frac=0.1)
print(sampled_data)

          country  year  gender          age  suicides_no  population  \
6207   Costa Rica  2001  female    75+ years            1       48980   
4496       Brazil  2012    male    75+ years          376     2123165   
16089       Malta  2008  female  35-54 years            2       54846   
13039       Italy  2005  female   5-14 years            2     2656241   
4379       Brazil  2002    male  25-34 years         1388    14524576   
...           ...   ...     ...          ...          ...         ...   
19666    Portugal  2013  female   5-14 years            0      518597   
10981   Guatemala  2009    male  35-54 years          109     1019866   
19312      Poland  2012    male  15-24 years          509     2533873   
14644  Kyrgyzstan  2003  female    75+ years            3       62686   
73        Albania  1995    male  55-74 years            9      178000   

       suicides/100k pop   gdp_for_year  gdp_per_capita  hdi_for_year  
6207                2.04    15913363335            

Validimi i vlerave duplikate <br>
Kontrollimi i kolonave specifike

In [21]:
duplicates_check=['country','year','gender','age']
duplicates=df.duplicated(subset=duplicates_check)

if duplicates.any():
    print("Duplicates found. Dropping duplicates.")
    df = df.drop_duplicates()
    print("\nCleaned DataFrame:")
    print(df)
else:
    print("No duplicates found. DataFrame remains unchanged.")

No duplicates found. DataFrame remains unchanged.


Kontrollimi i tërë dataframe-it


In [22]:
duplicates = df.duplicated()

if duplicates.any():
    print("Duplicates found.")
else:
    print("No duplicates found.")

No duplicates found.


Transformimi

In [23]:
# 1. Create a new column for total suicides per year
df['total_suicides'] = df.groupby('year')['suicides_no'].transform('sum')
# 5. Create a ratio of suicides to population
df['suicides_to_population_ratio'] = df['suicides_no'] / df['population']
print(df)

          country  year  gender          age  suicides_no  population  \
0         Albania  1987    male  15-24 years           21      312900   
1         Albania  1987    male  35-54 years           16      308000   
2         Albania  1987  female  15-24 years           14      289700   
3         Albania  1987    male    75+ years            1       21800   
4         Albania  1987    male  25-34 years            9      274300   
...           ...   ...     ...          ...          ...         ...   
27815  Uzbekistan  2014  female  35-54 years          107     3620833   
27816  Uzbekistan  2014  female    75+ years            9      348465   
27817  Uzbekistan  2014    male   5-14 years           60     2762158   
27818  Uzbekistan  2014  female   5-14 years           44     2631600   
27819  Uzbekistan  2014  female  55-74 years           21     1438935   

       suicides/100k pop  gdp_for_year  gdp_per_capita  hdi_for_year  \
0                   6.71    2156624900             

Diskretizimi i perpjestimit të vetëvrasjeve me numrin e popullesisë dhe i gdp në kategori më të përshtatshme

In [24]:
ratio_bins = [-1, 0, 1e-05, 2e-05, 4e-05, 6e-05, 8e-05, 1e-04]  
ratio_labels = ['None','Very Low', 'Low', 'Medium', 'High', 'Very High', 'Extreme']

df['suicides_to_population_ratio_discretize'] = pd.cut(df['suicides_to_population_ratio'], bins=ratio_bins, labels=ratio_labels)

gdp_bins = [0, 1000, 2000, 3000]
gdp_labels = ['Low', 'Medium', 'High']
df['gdp_category'] = pd.cut(df['gdp_per_capita'], bins=gdp_bins, labels=gdp_labels)

Binarizimi i kolones 'gender'

In [25]:
df['gender_encoded'] = df['gender'].map({'male': 1, 'female': 0})

Ruajtja e transformimeve ne nje file te ri

In [26]:

df.to_csv('cleaned_data.csv', index=False)